# LeetCode 1337 — The K Weakest Rows in a Matrix bằng Python

**Link đề:** https://leetcode.com/problems/the-k-weakest-rows-in-a-matrix/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix, Sorting, Binary Search, Heap (Priority Queue)

```python
class Solution:
    def kWeakestRows(self, mat: list[list[int]], k: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Làm quen **ma trận** (list lồng list): `mat[i]` là hàng, `mat[i][j]` là ô; `len(mat)`, `len(mat[0])`.
2. Sắp xếp theo **nhiều tiêu chí** bằng **tuple** `(số lính, chỉ số hàng)` — Python so sánh tuple theo thứ tự từ điển.
3. Đếm lính nhanh nhờ cấu trúc "1 đứng trước 0" → **binary search** `O(log n)` mỗi hàng.
4. Lấy `k` phần tử nhỏ nhất bằng **heap** (`heapq`) và cách **quét theo cột** sáng tạo.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho ma trận nhị phân `mat` kích thước `m × n`: `1` là **lính** (soldier), `0` là **dân thường** (civilian). Trong mỗi hàng, **mọi số 1 đứng bên trái mọi số 0**.

Hàng `i` **yếu hơn** hàng `j` nếu:
- số lính hàng `i` **ít hơn** hàng `j`, **hoặc**
- số lính bằng nhau và `i < j`.

Trả về chỉ số của `k` hàng **yếu nhất**, xếp từ yếu nhất đến mạnh hơn.

**Ví dụ 1:** `k = 3`
```text
hàng 0: [1,1,0,0,0] -> 2 lính
hàng 1: [1,1,1,1,0] -> 4
hàng 2: [1,0,0,0,0] -> 1
hàng 3: [1,1,0,0,0] -> 2
hàng 4: [1,1,1,1,1] -> 5
thứ tự yếu -> mạnh: [2,0,3,1,4]  =>  output [2,0,3]
```

**Ví dụ 2:** `mat = [[1,0,0,0],[1,1,1,1],[1,0,0,0],[1,0,0,0]]`, `k = 2` → `[0,2]` (hàng 0, 2, 3 cùng 1 lính; ưu tiên chỉ số nhỏ).

**Constraints:** `2 <= m, n <= 100`, `1 <= k <= m`, `mat[i][j]` là `0` hoặc `1`.

## 2. Nền tảng Python

### 2.1. Ma trận = list của list

In [1]:
mat = [[1, 1, 0, 0, 0],
       [1, 1, 1, 1, 0],
       [1, 0, 0, 0, 0],
       [1, 1, 0, 0, 0],
       [1, 1, 1, 1, 1]]
print("số hàng m =", len(mat), "| số cột n =", len(mat[0]))
print("hàng 1:", mat[1])
print("ô (1, 3):", mat[1][3])
for i, row in enumerate(mat):
    print("hàng", i, "->", row)

số hàng m = 5 | số cột n = 5
hàng 1: [1, 1, 1, 1, 0]
ô (1, 3): 1
hàng 0 -> [1, 1, 0, 0, 0]
hàng 1 -> [1, 1, 1, 1, 0]
hàng 2 -> [1, 0, 0, 0, 0]
hàng 3 -> [1, 1, 0, 0, 0]
hàng 4 -> [1, 1, 1, 1, 1]


### 2.2. Đếm lính trong một hàng: `sum(row)` hoặc `row.count(1)`

Vì hàng chỉ gồm 0 và 1, tổng các phần tử bằng số lính.

In [2]:
row = [1, 1, 0, 0, 0]
print(sum(row), row.count(1))
print([sum(r) for r in mat])

2 2
[2, 4, 1, 2, 5]


### 2.3. So sánh tuple — chìa khoá của "nhiều tiêu chí"

Python so sánh tuple **từng phần tử từ trái sang**: phần tử đầu khác nhau thì quyết định; bằng nhau thì xét phần tử tiếp theo.

`(2, 0) < (2, 3)` vì `2 == 2` rồi `0 < 3`. `(1, 2) < (2, 0)` vì `1 < 2`.

Vậy nếu mỗi hàng biểu diễn bằng tuple `(số_lính, chỉ_số)`, việc **sắp xếp tuple** cho đúng thứ tự "yếu → mạnh" theo định nghĩa của đề!

In [3]:
print((2, 0) < (2, 3))
print((1, 2) < (2, 0))
pairs = [(2, 0), (4, 1), (1, 2), (2, 3), (5, 4)]
print("sorted:", sorted(pairs))

True
True
sorted: [(1, 2), (2, 0), (2, 3), (4, 1), (5, 4)]


### 2.4. Slicing lấy `k` phần tử đầu: `lst[:k]`

`[2, 0, 3, 1, 4][:3]` → `[2, 0, 3]`.

### 2.5. `sorted(..., key=...)`

Thay vì tạo tuple, ta có thể sắp xếp **chỉ số** với `key` trả về tuple: `sorted(range(m), key=lambda i: (sum(mat[i]), i))`. Thực ra `sorted` là **ổn định** (stable): phần tử có key bằng nhau giữ nguyên thứ tự ban đầu, nên `key=lambda i: sum(mat[i])` cũng đủ (chỉ số đã tăng dần sẵn).

In [4]:
print([2, 0, 3, 1, 4][:3])
m = len(mat)
print(sorted(range(m), key=lambda i: (sum(mat[i]), i)))
print(sorted(range(m), key=lambda i: sum(mat[i])), "<- sorted ổn định, kết quả giống hệt")

[2, 0, 3]
[2, 0, 3, 1, 4]
[2, 0, 3, 1, 4] <- sorted ổn định, kết quả giống hệt


## 3. Cách trực giác / baseline — đếm rồi sắp xếp

1. Với mỗi hàng `i`, đếm lính `c = sum(mat[i])`, tạo tuple `(c, i)`.
2. Sắp xếp list tuple.
3. Lấy chỉ số của `k` tuple đầu.

In [5]:
def k_weakest_sort(mat: list[list[int]], k: int) -> list[int]:
    strength = [(sum(row), i) for i, row in enumerate(mat)]
    strength.sort()
    return [i for _, i in strength[:k]]

mat1 = [[1,1,0,0,0],[1,1,1,1,0],[1,0,0,0,0],[1,1,0,0,0],[1,1,1,1,1]]
mat2 = [[1,0,0,0],[1,1,1,1],[1,0,0,0],[1,0,0,0]]
print(k_weakest_sort(mat1, 3))
print(k_weakest_sort(mat2, 2))

[2, 0, 3]
[0, 2]


Đây đã là lời giải tốt cho constraints nhỏ: `O(m·n)` để đếm + `O(m log m)` để sắp xếp.

Có thể cải thiện ở **hai chỗ**:
1. **Đếm lính**: hàng có dạng `1...10...0` — đã **sắp xếp giảm dần**! Không cần duyệt cả hàng, dùng **binary search** tìm vị trí số 0 đầu tiên: `O(log n)`.
2. **Sắp xếp**: ta chỉ cần `k` hàng yếu nhất, không cần sắp xếp hết → **heap** `O(m log k)`.

## 4. Insight 1 — binary search đếm lính

Trong hàng `[1,1,1,0,0]`, số lính = **chỉ số của số 0 đầu tiên** (= 3). Nếu không có 0 thì = `n`.

Binary search: giữ khoảng `[lo, hi)` chứa vị trí cần tìm. Xét `mid`: nếu `row[mid] == 1` thì số 0 đầu tiên nằm **bên phải** `mid` → `lo = mid + 1`; ngược lại nằm ở `mid` hoặc bên trái → `hi = mid`. Dừng khi `lo == hi`.

In [6]:
def count_soldiers(row: list[int]) -> int:
    lo, hi = 0, len(row)
    while lo < hi:
        mid = (lo + hi) // 2
        if row[mid] == 1:
            lo = mid + 1
        else:
            hi = mid
    return lo

for row in [[1,1,1,0,0], [0,0,0], [1,1,1], [1,0,0,0,0]]:
    print(row, "->", count_soldiers(row), "| kiểm bằng sum:", sum(row))

[1, 1, 1, 0, 0] -> 3 | kiểm bằng sum: 3
[0, 0, 0] -> 0 | kiểm bằng sum: 0
[1, 1, 1] -> 3 | kiểm bằng sum: 3
[1, 0, 0, 0, 0] -> 1 | kiểm bằng sum: 1


In [7]:
def trace_count_soldiers(row):
    lo, hi = 0, len(row)
    print("row =", row)
    while lo < hi:
        mid = (lo + hi) // 2
        go = "phải (lo = mid+1)" if row[mid] == 1 else "trái (hi = mid)"
        print(f"  lo={lo}, hi={hi}, mid={mid}, row[mid]={row[mid]} -> sang {go}")
        if row[mid] == 1:
            lo = mid + 1
        else:
            hi = mid
    print(f"  dừng: lo = hi = {lo} -> {lo} lính")
    return lo

trace_count_soldiers([1, 1, 1, 1, 0, 0, 0, 0])

row = [1, 1, 1, 1, 0, 0, 0, 0]
  lo=0, hi=8, mid=4, row[mid]=0 -> sang trái (hi = mid)
  lo=0, hi=4, mid=2, row[mid]=1 -> sang phải (lo = mid+1)
  lo=3, hi=4, mid=3, row[mid]=1 -> sang phải (lo = mid+1)
  dừng: lo = hi = 4 -> 4 lính


4

## 5. Lời giải chính — binary search + sắp xếp tuple

**Đoạn 1:** `strength = [(count_soldiers(row), i) for i, row in enumerate(mat)]`.

**Đoạn 2:** `strength.sort()` — tuple tự lo cả hai tiêu chí.

**Đoạn 3:** trả `[i for _, i in strength[:k]]`.

In [8]:
def k_weakest_bs(mat: list[list[int]], k: int) -> list[int]:
    strength = [(count_soldiers(row), i) for i, row in enumerate(mat)]
    strength.sort()
    return [i for _, i in strength[:k]]

print(k_weakest_bs(mat1, 3))
print(k_weakest_bs(mat2, 2))

[2, 0, 3]
[0, 2]


## 6. TRACE từng bước

Trace ví dụ 1, `k = 3`:

| hàng i | row | số lính (binary search) | tuple |
|---|---|---|---|
| 0 | `[1,1,0,0,0]` | 2 | `(2,0)` |
| 1 | `[1,1,1,1,0]` | 4 | `(4,1)` |
| 2 | `[1,0,0,0,0]` | 1 | `(1,2)` |
| 3 | `[1,1,0,0,0]` | 2 | `(2,3)` |
| 4 | `[1,1,1,1,1]` | 5 | `(5,4)` |

Sau sắp xếp: `(1,2), (2,0), (2,3), (4,1), (5,4)` → 3 đầu → `[2, 0, 3]`. Chú ý `(2,0)` đứng trước `(2,3)`: cùng 2 lính, chỉ số nhỏ hơn yếu hơn.

In [9]:
def trace_k_weakest(mat, k):
    strength = []
    for i, row in enumerate(mat):
        c = count_soldiers(row)
        strength.append((c, i))
        print(f"  hàng {i}: {row} -> {c} lính -> tuple {(c, i)}")
    strength.sort()
    print("  sau sort:", strength)
    ans = [i for _, i in strength[:k]]
    print(f"  lấy {k} đầu -> {ans}")
    return ans

trace_k_weakest(mat1, 3)
print()
trace_k_weakest(mat2, 2)

  hàng 0: [1, 1, 0, 0, 0] -> 2 lính -> tuple (2, 0)
  hàng 1: [1, 1, 1, 1, 0] -> 4 lính -> tuple (4, 1)
  hàng 2: [1, 0, 0, 0, 0] -> 1 lính -> tuple (1, 2)
  hàng 3: [1, 1, 0, 0, 0] -> 2 lính -> tuple (2, 3)
  hàng 4: [1, 1, 1, 1, 1] -> 5 lính -> tuple (5, 4)
  sau sort: [(1, 2), (2, 0), (2, 3), (4, 1), (5, 4)]
  lấy 3 đầu -> [2, 0, 3]

  hàng 0: [1, 0, 0, 0] -> 1 lính -> tuple (1, 0)
  hàng 1: [1, 1, 1, 1] -> 4 lính -> tuple (4, 1)
  hàng 2: [1, 0, 0, 0] -> 1 lính -> tuple (1, 2)
  hàng 3: [1, 0, 0, 0] -> 1 lính -> tuple (1, 3)
  sau sort: [(1, 0), (1, 2), (1, 3), (4, 1)]
  lấy 2 đầu -> [0, 2]


[0, 2]

## 7. Code nộp LeetCode

In [10]:
class Solution:
    def kWeakestRows(self, mat: list[list[int]], k: int) -> list[int]:
        def soldiers(row: list[int]) -> int:
            lo, hi = 0, len(row)
            while lo < hi:
                mid = (lo + hi) // 2
                if row[mid] == 1:
                    lo = mid + 1
                else:
                    hi = mid
            return lo

        strength = sorted((soldiers(row), i) for i, row in enumerate(mat))
        return [i for _, i in strength[:k]]

sol = Solution()
print(sol.kWeakestRows(mat1, 3), sol.kWeakestRows(mat2, 2))

[2, 0, 3] [0, 2]


## 8. Các lời giải khác

### Cách 3 — heap: chỉ giữ `k` hàng yếu nhất

Module chuẩn `heapq` cài đặt **min-heap** trên list: `heapq.heappush(h, x)` thêm, `heapq.heappop(h)` lấy phần tử **nhỏ nhất**, đều `O(log size)`.

Mẹo: muốn giữ `k` phần tử **nhỏ nhất**, dùng **max-heap kích thước `k`** (phần tử lớn nhất ở đỉnh để loại khi có phần tử nhỏ hơn). Python chỉ có min-heap → lưu tuple **đổi dấu** `(-c, -i)`.

Cách ngắn hơn: `heapq.nsmallest(k, iterable)` làm sẵn việc này.

In [11]:
import heapq

h = []
for x in [5, 1, 4, 2]:
    heapq.heappush(h, x)
    print("push", x, "-> heap (list nội bộ):", h, "| nhỏ nhất h[0] =", h[0])
print("pop ->", heapq.heappop(h), "| còn lại:", h)
print("nsmallest(2, [5,1,4,2]) =", heapq.nsmallest(2, [5, 1, 4, 2]))

push 5 -> heap (list nội bộ): [5] | nhỏ nhất h[0] = 5
push 1 -> heap (list nội bộ): [1, 5] | nhỏ nhất h[0] = 1
push 4 -> heap (list nội bộ): [1, 5, 4] | nhỏ nhất h[0] = 1
push 2 -> heap (list nội bộ): [1, 2, 4, 5] | nhỏ nhất h[0] = 1
pop -> 1 | còn lại: [2, 5, 4]
nsmallest(2, [5,1,4,2]) = [1, 2]


In [12]:
def k_weakest_heap(mat: list[list[int]], k: int) -> list[int]:
    heap = []                                  # max-heap kích thước k (lưu số âm)
    for i, row in enumerate(mat):
        item = (-count_soldiers(row), -i)
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:                   # hàng i yếu hơn hàng "mạnh nhất" trong heap
            heapq.heapreplace(heap, item)      # pop đỉnh rồi push item
    result = []
    while heap:
        c, i = heapq.heappop(heap)             # lấy ra từ mạnh -> yếu
        result.append(-i)
    return result[::-1]                        # đảo để yếu -> mạnh

def k_weakest_nsmallest(mat, k):
    return [i for _, i in heapq.nsmallest(k, ((count_soldiers(r), i) for i, r in enumerate(mat)))]

print(k_weakest_heap(mat1, 3), k_weakest_heap(mat2, 2))
print(k_weakest_nsmallest(mat1, 3), k_weakest_nsmallest(mat2, 2))

[2, 0, 3] [0, 2]
[2, 0, 3] [0, 2]


Vì sao `item > heap[0]` nghĩa là "yếu hơn"? `heap[0]` là tuple âm **nhỏ nhất** = hàng **mạnh nhất** trong heap. `(-c, -i) > (-c0, -i0)` ⇔ `(c, i) < (c0, i0)` ⇔ hàng `i` yếu hơn.

### Cách 4 — quét theo **cột** (sáng tạo, không cần đếm)

Đi từng cột `j = 0, 1, ...`, trong mỗi cột duyệt hàng từ trên xuống. Hàng `i` có `mat[i][j] == 0` **lần đầu tiên** ở cột `j` nghĩa là nó có đúng `j` lính. Các hàng "chết" ở cột sớm hơn thì yếu hơn; cùng cột thì hàng trên (chỉ số nhỏ) được duyệt trước → đúng thứ tự! Cuối cùng các hàng toàn `1` thêm vào sau.

Dừng ngay khi đủ `k` hàng.

In [13]:
def k_weakest_columns(mat: list[list[int]], k: int) -> list[int]:
    m, n = len(mat), len(mat[0])
    result = []
    added = [False] * m
    for j in range(n):
        for i in range(m):
            if not added[i] and mat[i][j] == 0:
                result.append(i)
                added[i] = True
                if len(result) == k:
                    return result
    for i in range(m):                     # các hàng toàn lính
        if not added[i]:
            result.append(i)
            if len(result) == k:
                return result
    return result

print(k_weakest_columns(mat1, 3), k_weakest_columns(mat2, 2))

[2, 0, 3] [0, 2]


### Code nộp LeetCode cho các cách khác

In [14]:
class SolutionSimpleSort:
    def kWeakestRows(self, mat: list[list[int]], k: int) -> list[int]:
        return sorted(range(len(mat)), key=lambda i: (sum(mat[i]), i))[:k]

class SolutionHeap:
    def kWeakestRows(self, mat: list[list[int]], k: int) -> list[int]:
        return [i for _, i in heapq.nsmallest(k, ((sum(r), i) for i, r in enumerate(mat)))]

class SolutionColumns:
    def kWeakestRows(self, mat: list[list[int]], k: int) -> list[int]:
        m, n = len(mat), len(mat[0])
        order, done = [], [False] * m
        for j in range(n + 1):                 # cột "n" ảo: hàng toàn lính
            for i in range(m):
                if not done[i] and (j == n or mat[i][j] == 0):
                    order.append(i)
                    done[i] = True
        return order[:k]

for cls in [SolutionSimpleSort, SolutionHeap, SolutionColumns]:
    print(cls.__name__, cls().kWeakestRows(mat1, 3), cls().kWeakestRows(mat2, 2))

SolutionSimpleSort [2, 0, 3] [0, 2]
SolutionHeap [2, 0, 3] [0, 2]
SolutionColumns [2, 0, 3] [0, 2]


### Bảng so sánh

| Cách | Đếm lính | Chọn k | Time | Space |
|---|---|---|---|---|
| 1. `sum` + sort | `O(n)`/hàng | sort tất cả | `O(m·n + m log m)` | `O(m)` |
| 2. Binary search + sort | `O(log n)`/hàng | sort tất cả | `O(m log n + m log m)` | `O(m)` |
| 3. Binary search + heap k | `O(log n)`/hàng | heap kích thước k | `O(m log n + m log k)` | `O(k)` |
| 4. Quét cột | không đếm | dừng sớm | `O(m·n)` xấu nhất | `O(m)` |

Với `m, n <= 100`, mọi cách đều nhanh. Cách 2 là "chuẩn phỏng vấn": khai thác cấu trúc sắp xếp của hàng.

## 9. Vì sao đúng?

1. **Đếm lính đúng (binary search):** invariant — mọi vị trí `< lo` chứa `1`, mọi vị trí `>= hi` chứa `0` (hoặc ngoài mảng). Ban đầu `lo = 0`, `hi = n` đúng hiển nhiên. Mỗi bước giữ invariant và thu hẹp `[lo, hi)`. Khi `lo == hi`, đó là vị trí số 0 đầu tiên = số lính.
2. **Thứ tự đúng:** định nghĩa "yếu hơn" chính là so sánh tuple `(số lính, chỉ số)` theo thứ tự từ điển. Sắp xếp tăng dần các tuple cho thứ tự yếu → mạnh. Các tuple đôi một khác nhau (chỉ số khác nhau) nên thứ tự là duy nhất.
3. Lấy `k` phần tử đầu là `k` hàng yếu nhất. ∎

### 9.1. Độ phức tạp (lời giải chính)

`m` hàng, `n` cột.
- **Time:** `O(m log n)` đếm + `O(m log m)` sắp xếp = `O(m (log n + log m))`.
- **Space:** `O(m)` cho list tuple.

## 10. Bộ test

Mẫu đề, `k = m`, hàng toàn 0 / toàn 1, nhiều hàng bằng nhau, và **đối chiếu ngẫu nhiên** mọi cách.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS -> {actual}")

check("ví dụ 1", sol.kWeakestRows(mat1, 3), [2, 0, 3])
check("ví dụ 2", sol.kWeakestRows(mat2, 2), [0, 2])
check("k = m", sol.kWeakestRows(mat1, 5), [2, 0, 3, 1, 4])
check("toàn 0 và toàn 1", sol.kWeakestRows([[1, 1], [0, 0], [1, 1], [0, 0]], 3), [1, 3, 0])
check("tất cả bằng nhau", sol.kWeakestRows([[1, 0]] * 4, 2), [0, 1])
check("k = 1", sol.kWeakestRows([[1, 1, 1], [1, 1, 0]], 1), [1])

ví dụ 1              PASS -> [2, 0, 3]
ví dụ 2              PASS -> [0, 2]
k = m                PASS -> [2, 0, 3, 1, 4]
toàn 0 và toàn 1     PASS -> [1, 3, 0]
tất cả bằng nhau     PASS -> [0, 1]
k = 1                PASS -> [1]


In [16]:
import random
rng = random.Random(1337)

def random_mat(m, n):
    return [[1] * c + [0] * (n - c) for c in (rng.randint(0, n) for _ in range(m))]

methods = [k_weakest_sort, k_weakest_bs, k_weakest_heap, k_weakest_nsmallest, k_weakest_columns,
           sol.kWeakestRows, SolutionSimpleSort().kWeakestRows, SolutionHeap().kWeakestRows,
           SolutionColumns().kWeakestRows]
for _ in range(500):
    m, n = rng.randint(2, 12), rng.randint(2, 8)
    mat = random_mat(m, n)
    k = rng.randint(1, m)
    expected = k_weakest_sort(mat, k)
    for f in methods:
        assert f(mat, k) == expected, (f, mat, k)
for row_len in range(0, 30):
    for c in range(row_len + 1):
        assert count_soldiers([1] * c + [0] * (row_len - c)) == c
print("500 ma trận ngẫu nhiên: mọi cách khớp; binary search đếm đúng mọi hàng dài 0..29")

500 ma trận ngẫu nhiên: mọi cách khớp; binary search đếm đúng mọi hàng dài 0..29


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chỉ sắp xếp theo số lính, rồi lấy **giá trị** thay vì chỉ số

In [17]:
def wrong_return_counts(mat, k):
    return sorted(sum(r) for r in mat)[:k]       # trả số lính, không phải chỉ số hàng

print("sai :", wrong_return_counts(mat1, 3), "| đúng:", sol.kWeakestRows(mat1, 3))

sai : [1, 2, 2] | đúng: [2, 0, 3]


### Lỗi 2: Dùng `dict` số lính → hàng (mất hàng trùng số lính)

In [18]:
def wrong_dict(mat, k):
    d = {sum(r): i for i, r in enumerate(mat)}    # key trùng bị ghi đè!
    return [d[c] for c in sorted(d)][:k]

print("sai :", wrong_dict(mat1, 3), "| đúng:", sol.kWeakestRows(mat1, 3))

sai : [2, 3, 1] | đúng: [2, 0, 3]


### Lỗi 3: Binary search sai biên → vòng lặp vô hạn

Viết `hi = len(row) - 1` và `while lo <= hi` nhưng cập nhật `hi = mid` (thay vì `mid - 1`) có thể kẹt mãi khi `lo == hi`. Chọn **một** quy ước (`[lo, hi)` nửa mở) và giữ nhất quán.

### Lỗi 4: Max-heap với Python mà quên đổi dấu chỉ số

Dùng `(-c, i)` thay vì `(-c, -i)`: khi số lính bằng nhau, heap coi hàng chỉ số **nhỏ** là "mạnh hơn" → bị loại nhầm.

In [19]:
def wrong_heap_sign(mat, k):
    heap = []
    for i, row in enumerate(mat):
        item = (-sum(row), i)                     # sai: phải là -i
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)
    return [i for _, i in sorted(heap, key=lambda t: (-t[0], t[1]))]

m3 = [[1, 0], [1, 0], [1, 0], [1, 1]]
print("sai :", wrong_heap_sign(m3, 2), "| đúng:", sol.kWeakestRows(m3, 2))

sai : [1, 2] | đúng: [0, 1]


## 12. Mẹo nhận dạng pattern

- "Sắp xếp theo A, hoà thì theo B" ⇒ **key là tuple `(A, B)`** hoặc dựa vào `sorted` ổn định.
- Hàng/mảng **đã sắp xếp** (1 trước 0) ⇒ đếm bằng **binary search** (`bisect` hoặc tự viết).
- "Top k nhỏ nhất/lớn nhất" ⇒ **heap kích thước k** hoặc `heapq.nsmallest/nlargest`.
- Ma trận có cấu trúc đơn điệu theo hàng ⇒ thử **quét theo cột**.

## 13. Biến thể

1. **K hàng mạnh nhất** (nhiều lính nhất, hoà thì chỉ số nhỏ trước): key `(-lính, i)`.
2. **Đếm số 0 trong hàng sắp xếp tăng** (0 trước 1): binary search vị trí số 1 đầu tiên.
3. **LeetCode 1351 — Count Negative Numbers in a Sorted Matrix**: cùng ý tưởng binary search / quét bậc thang.
4. Dùng thư viện: hàng `1...10...0` đảo dấu thành tăng dần → `bisect`.

In [20]:
from bisect import bisect_left

def k_strongest(mat, k):
    return [i for _, i in sorted((-sum(r), i) for i, r in enumerate(mat))[:k]]

def soldiers_bisect(row):
    # đổi 1 -> -1 để hàng thành tăng dần: [-1,-1,0,0]; vị trí số 0 đầu = bisect_left(..., 0)
    return bisect_left([-x for x in row], 0)

print("mạnh nhất:", k_strongest(mat1, 2))
print([soldiers_bisect(r) for r in mat1])

mạnh nhất: [4, 1]
[2, 4, 1, 2, 5]


## 14. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** `mat = [[1,1,1],[1,0,0],[1,1,0],[1,0,0]]`, `k = 3` → output?

**Bài 2 (đoán output):** `sorted([(2, 3), (1, 5), (2, 1)])`, `(3, 0) < (2, 9)`.

**Bài 3 (tự viết):** `exercise_count(row)` — binary search đếm số 1 trong hàng `1...10...0`.

**Bài 4 (tự viết):** `exercise_k_weakest(mat, k)` dùng `exercise_count` và tuple.

**Bài 5 (biến thể):** `exercise_k_strongest(mat, k)`.

**Bài 6 (mini challenge):** `exercise_k_weakest_heap(mat, k)` — dùng heap kích thước tối đa `k` (không dùng `nsmallest`, không sort toàn bộ).

In [21]:
def exercise_count(row: list[int]) -> int:
    # TODO
    pass

def exercise_k_weakest(mat: list[list[int]], k: int) -> list[int]:
    # TODO
    pass

# assert exercise_count([1, 1, 0]) == 2 and exercise_count([0, 0]) == 0 and exercise_count([1, 1]) == 2
# assert exercise_k_weakest(mat1, 3) == [2, 0, 3]

In [22]:
def exercise_k_strongest(mat: list[list[int]], k: int) -> list[int]:
    # TODO
    pass

def exercise_k_weakest_heap(mat: list[list[int]], k: int) -> list[int]:
    # TODO
    pass

# assert exercise_k_strongest(mat1, 2) == [4, 1]
# assert exercise_k_weakest_heap(mat1, 3) == [2, 0, 3]

## 15. Gợi ý 3 mức

**Bài 1** — (1) Đếm lính: 3, 1, 2, 1. (2) Tuple: `(3,0),(1,1),(2,2),(1,3)`. (3) Sắp xếp và lấy 3 chỉ số đầu.

**Bài 3** — (1) Khoảng `[lo, hi)` với `hi = len(row)`. (2) `row[mid] == 1` → `lo = mid + 1`. (3) Ngược lại `hi = mid`; trả `lo`.

**Bài 5** — (1) Đảo tiêu chí lính, giữ tiêu chí chỉ số. (2) Key `(-lính, i)`. (3) `sorted(...)[:k]`.

**Bài 6** — (1) Max-heap = min-heap với số âm. (2) Item `(-lính, -i)`. (3) Khi heap đủ k, chỉ thay đỉnh nếu item mới **lớn hơn** đỉnh; cuối cùng pop hết và đảo ngược.

## 16. Đáp án tham khảo

**Bài 1:** `[1, 3, 2]`.

**Bài 2:** `[(1, 5), (2, 1), (2, 3)]`, `False`.

In [23]:
def answer_count(row):
    lo, hi = 0, len(row)
    while lo < hi:
        mid = (lo + hi) // 2
        if row[mid] == 1:
            lo = mid + 1
        else:
            hi = mid
    return lo

def answer_k_weakest(mat, k):
    return [i for _, i in sorted((answer_count(r), i) for i, r in enumerate(mat))[:k]]

def answer_k_strongest(mat, k):
    return [i for _, i in sorted((-answer_count(r), i) for i, r in enumerate(mat))[:k]]

def answer_k_weakest_heap(mat, k):
    heap = []
    for i, r in enumerate(mat):
        item = (-answer_count(r), -i)
        if len(heap) < k:
            heapq.heappush(heap, item)
        elif item > heap[0]:
            heapq.heapreplace(heap, item)
    out = []
    while heap:
        out.append(-heapq.heappop(heap)[1])
    return out[::-1]

assert answer_k_weakest([[1,1,1],[1,0,0],[1,1,0],[1,0,0]], 3) == [1, 3, 2]
assert sorted([(2, 3), (1, 5), (2, 1)]) == [(1, 5), (2, 1), (2, 3)] and ((3, 0) < (2, 9)) is False
assert answer_count([1, 1, 0]) == 2 and answer_count([0, 0]) == 0 and answer_count([1, 1]) == 2
assert answer_k_weakest(mat1, 3) == [2, 0, 3]
assert answer_k_strongest(mat1, 2) == [4, 1]
assert answer_k_weakest_heap(mat1, 3) == [2, 0, 3]
for _ in range(300):
    m, n = rng.randint(2, 10), rng.randint(2, 6)
    mat = random_mat(m, n); k = rng.randint(1, m)
    assert answer_k_weakest_heap(mat, k) == answer_k_weakest(mat, k) == k_weakest_sort(mat, k)
    assert answer_k_strongest(mat, k) == sorted(range(m), key=lambda i: (-sum(mat[i]), i))[:k]
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def kWeakestRows(self, mat: list[list[int]], k: int) -> list[int]:
        def soldiers(row):                      # binary search vị trí số 0 đầu tiên
            lo, hi = 0, len(row)
            while lo < hi:
                mid = (lo + hi) // 2
                if row[mid] == 1: lo = mid + 1
                else: hi = mid
            return lo
        return [i for _, i in sorted((soldiers(r), i) for i, r in enumerate(mat))[:k]]
```

**Checklist:**
- [ ] Trả **chỉ số hàng**, không phải số lính.
- [ ] Hoà số lính → chỉ số nhỏ trước (tuple `(lính, i)`).
- [ ] Binary search dùng nhất quán khoảng `[lo, hi)`.
- [ ] Đã test hàng toàn 0, toàn 1, nhiều hàng bằng nhau, `k = m`.